# Colibrí Intelligence · demo de punta a punta

Este notebook usa el sistema **levantado** (`./run.sh` o `docker compose up -d`) a través de su API, como lo haría cualquier cliente:

1. **RAG directo** (`/ask`): una pregunta puntual a la base de conocimiento, con fuentes verificadas.
2. **Streaming** (`/ask/stream`): la misma idea, palabra por palabra.
3. **Equipo de agentes** (`/tasks`): un análisis de varios pasos coordinado por el supervisor.
4. **Aprobación humana**: una acción cara que se frena hasta que una persona la aprueba.
5. **Memoria de la conversación**: un segundo pedido que solo se entiende con lo que pasó antes.

Cada paso deja su traza en Phoenix (http://127.0.0.1:6006, proyecto `colibri-intelligence`).

In [1]:
import json
import time

import httpx

URL = "http://127.0.0.1:8000"   # 127.0.0.1 y no localhost: en Windows evita ~300 ms por conexión
api = httpx.Client(base_url=URL, timeout=120)
print(json.dumps(api.get("/health").json(), indent=2, ensure_ascii=False))

{
  "redis": true,
  "base_vectorial": {
    "coleccion": "colibri-conocimiento",
    "fragmentos": 35,
    "embeddings": "google"
  },
  "workers_en_api": 0,
  "modelos": "llm",
  "trazas": "http://phoenix:6006/v1/traces (proyecto colibri-intelligence)"
}


## 1. RAG directo: pregunta puntual con fuentes verificadas

In [2]:
def preguntar(pregunta: str) -> None:
    inicio = time.perf_counter()
    r = api.post("/ask", json={"pregunta": pregunta}).json()
    print(f"P: {pregunta}  ({time.perf_counter() - inicio:.1f} s)")
    print(f"R: {r['respuesta']}")
    for f in r["fuentes"]:
        print(f"   fuente: {f['documento_id']}  ({f['titulo']} > {f['seccion']})")
    print(f"   recuperados: {r['recuperados']}\n")


preguntar("¿Qué tiene que activar la guardia si el proveedor de pagos responde con errores?")
preguntar("Si alguien pega una contraseña en un chat, ¿qué hay que hacer?")
preguntar("¿Cuál es la capital de Australia?")   # no está en la base: tiene que decir "No lo sé." 

P: ¿Qué tiene que activar la guardia si el proveedor de pagos responde con errores?  (6.7 s)
R: Si el problema es del proveedor de pagos, la guardia debe activar el feature flag pagos_modo_cola.
   fuente: guia_incidentes.md#runbook-errores-5xx-en-pagos  (Guía de incidentes de Colibrí > Runbook: errores 5xx en pagos)
   recuperados: ['guia_incidentes.md#runbook-errores-5xx-en-pagos::0', 'postmortem_pagos_2026-10.md#resumen-del-incidente::0', 'postmortem_pagos_2026-10.md#causa-raiz::0', 'guia_incidentes.md#guardia-on-call::0']



P: Si alguien pega una contraseña en un chat, ¿qué hay que hacer?  (3.9 s)
R: Si un secreto se filtra (por ejemplo, alguien lo pega en un chat), se considera comprometido y se debe rotar de inmediato, incluso si no hay señales de que alguien lo haya usado.
   fuente: seguridad.md#gestion-de-secretos  (Políticas de seguridad de Colibrí > Gestión de secretos)
   recuperados: ['seguridad.md#gestion-de-secretos::0', 'seguridad.md#acceso-a-produccion::0', 'faq.json#faq-07::0', 'faq.json#faq-08::0']



P: ¿Cuál es la capital de Australia?  (4.2 s)
R: No lo sé.
   recuperados: ['faq.json#faq-08::0', 'faq.json#faq-02::0', 'arquitectura.md#vision-general::0', 'faq.json#faq-07::0']



La última pregunta no está en la base de conocimiento: el sistema responde **"No lo sé."** en lugar de inventar. Las fuentes que cita el modelo se comparan con lo que se recuperó de verdad: si cita algo que no se le pasó, se descarta.

## 2. Streaming: la respuesta llega palabra por palabra (Server-Sent Events)

In [3]:
inicio = time.perf_counter()
with api.stream("POST", "/ask/stream", json={"pregunta": "¿Con qué comando se pasa Redis a la réplica?"}) as r:
    evento = None
    for linea in r.iter_lines():
        if linea.startswith("event: "):
            evento = linea.removeprefix("event: ")
        elif linea.startswith("data: "):
            datos = json.loads(linea.removeprefix("data: "))
            if evento == "fuentes":
                print(f"[{time.perf_counter() - inicio:.2f} s] fuentes: {[f['documento_id'] for f in datos]}")
            elif evento == "token":
                print(datos, end="", flush=True)
            elif evento == "fin":
                print(f"\n[{time.perf_counter() - inicio:.2f} s] fin: {datos}")

[0.41 s] fuentes: ['guia_incidentes.md#runbook-redis-caido-o-inaccesible', 'arquitectura.md#datos', 'arquitectura.md#flujo-de-una-compra', 'seguridad.md#logs']


Para pasar

 Redis a la réplica, debes ejecutar el comando `colibri-ops redis failover`.



[guia_incidentes.md#runbook-redis-caido-o-inaccesible::0]


[4.75 s] fin: {'encontrada': True, 'fuentes_citadas': ['guia_incidentes.md#runbook-redis-caido-o-inaccesible::0']}


## 3. Una tarea compleja para el equipo de agentes

`POST /tasks` responde al instante con un `job_id`; el trabajo lo hacen los workers. El cliente consulta el estado (polling) y ve por qué nodos va pasando el grafo.

In [4]:
def seguir(job_id: str, hasta=("DONE", "FAILED", "WAITING_APPROVAL")) -> dict:
    visto = 0
    while True:
        job = api.get(f"/tasks/{job_id}").json()
        if len(job["recorrido"]) > visto:
            print(" -> ".join(job["recorrido"][visto:]), end=" -> ", flush=True)
            visto = len(job["recorrido"])
        if job["estado"] in hasta:
            print(f"[{job['estado']}] {job.get('duracion_s')} s de proceso")
            return job
        time.sleep(1)


inicio = time.perf_counter()
encolado = api.post("/tasks", json={"pedido": "¿Cómo impactó el incidente de pagos de octubre en la opinión de los clientes? "
                                              "Quiero saber qué pasó, cuánto cambiaron las reseñas y qué deberíamos hacer."}).json()
print(f"POST /tasks respondió en {(time.perf_counter() - inicio) * 1000:.0f} ms: {encolado}\n")
analisis = seguir(encolado["job_id"])

POST /tasks respondió en 8 ms: {'job_id': '0f8cf5fb-6a0c-4ccc-a2e5-eee1127675bd', 'conversacion_id': '5d0aeffc-ec8a-4c7c-b25e-1b66d4b52389', 'estado': 'PENDING', 'consultar': '/tasks/0f8cf5fb-6a0c-4ccc-a2e5-eee1127675bd'}



supervisor -> 

investigador -> 

supervisor -> 

analista -> 

supervisor -> 

redactor -> validador -> 

supervisor -> 

[DONE] 41.395 s de proceso


In [5]:
informe = analisis["resultado"]["informe"]
print(f"# {informe['titulo']}\n\n{informe['resumen']}\n")
print("Hallazgos:", *informe["hallazgos"], sep="\n - ")
print("\nRecomendaciones:", *informe["recomendaciones"], sep="\n - ")
print("\nFuentes:", *informe["fuentes"], sep="\n - ")
print(f"\nValidado por el validador determinista: {analisis['resultado']['validado']}")

# Impacto del incidente de pagos de octubre 2026 en la satisfacción del cliente

El incidente del 14 de octubre, causado por una falla en el proveedor externo de pagos, provocó una caída significativa en la satisfacción del cliente tras errores en los cobros y demoras. La calificación promedio descendió de 4.25 a 2.00, con un aumento drástico en las reseñas negativas que pasaron de representar el 0% al 71.43% del total.

Hallazgos:
 - La caída del proveedor externo de pagos generó una tasa de errores 5xx del 38%, afectando a 1.240 pedidos y provocando 312 cobros dobles debido a la falta de claves de idempotencia.
 - El impacto en los clientes se agravó por una respuesta técnica tardía y la acumulación de demoras en notificaciones y soporte, que cuadruplicó sus contactos la semana posterior.
 - La calificación promedio cayó un 52.94% (de 4.25 a 2.00) y las reseñas con baja calificación (1 o 2 estrellas) aumentaron 71.43 puntos porcentuales respecto a la semana previa al incidente.

Reco

### Quién hizo qué

Cada agente deja un aporte en el estado compartido: es el registro del razonamiento del supervisor y del trabajo de cada especialista.

In [6]:
for aporte in analisis["resultado"]["aportes"]:
    print(f"{aporte['paso']:>2}. {aporte['agente']:<12} {aporte['resumen'][:160]}")

 1. supervisor   -> investigador: Busca la documentación del incidente de pagos de octubre y extrae las reseñas de clientes de los dos períodos: antes y después del incidente, u
 2. investigador Encontré el postmortem del incidente ocurrido el 14/10/2026, donde una falla del proveedor de pagos y problemas técnicos internos (ausencia de idempotencia y de
 3. supervisor   -> analista: Calcula las métricas y el sentimiento de las reseñas para los dos períodos (07/10/2026 al 13/10/2026 y 15/10/2026 al 21/10/2026) y compara los resu
 4. analista     El incidente de pagos provocó un impacto severo: la calificación promedio cayó un 52.94% (de 4.25 a 2.00) y el sentimiento negativo se disparó un 71.43%. Para r
 5. supervisor   -> redactor: Redacta el informe final detallando el impacto del incidente de pagos de octubre en las reseñas y métricas comparativas entre los dos períodos, jun
 6. redactor     Informe 'Impacto del incidente de pagos de octubre 2026 en la satisfacción del cliente' con 6

## 4. Una acción cara espera la aprobación de una persona

Mandar un cupón del 10% a los 312 clientes con cobro doble cuesta más que el umbral ($50.000): el grafo se pausa con `interrupt()`, el estado queda en Redis y el job pasa a `WAITING_APPROVAL`.

In [7]:
accion = api.post("/tasks", json={"pedido": "Mandá un cupón del 10% a los clientes con cobro doble por el incidente de pagos."}).json()
pausado = seguir(accion["job_id"])
print("\nPregunta para la persona:", pausado["pregunta"]["pregunta"])

supervisor -> 

investigador -> 

supervisor -> 

operador -> 

[WAITING_APPROVAL] 29.045 s de proceso

Pregunta para la persona: ¿Aprobás esta acción? Enviar un cupón del 10% a 312 clientes a los que se les cobró dos veces durante el incidente de pagos del 14/10/2026. Costo estimado: $212.160.


In [8]:
decision = {"aprobado": True, "revisor": "Roger", "comentario": "Es lo que recomienda el postmortem."}
print(api.post(f"/tasks/{accion['job_id']}/approve", json=decision).json())
terminado = seguir(accion["job_id"], hasta=("DONE", "FAILED"))
print("\nResultado de la acción:", terminado["resultado"]["resultado_accion"])

{'job_id': '5d2bfb0d-de0d-4f2e-828e-67928d3914ce', 'conversacion_id': '53e6bdd3-21ad-4fdb-8fe9-188bec358930', 'estado': 'PENDING', 'consultar': '/tasks/5d2bfb0d-de0d-4f2e-828e-67928d3914ce'}
supervisor -> investigador -> supervisor -> operador -> 

aprobacion_humana -> ejecutor -> 

supervisor -> 

redactor -> validador -> 

supervisor -> 

redactor -> validador -> 

supervisor -> 

[DONE] 58.802 s de proceso

Resultado de la acción: Se enviaron 312 cupones del 10% (costo estimado $212.160), aprobados por Roger.


## 5. Memoria de la conversación

El segundo pedido va a la **misma conversación** (`conversacion_id`). No dice de qué acción habla: el supervisor lo resuelve con el turno anterior, que está guardado en el checkpointer de Redis (sobrevive a un reinicio).

In [9]:
seguimiento = api.post("/tasks", json={"pedido": "¿Qué acción se tomó con esos clientes y cuánto costó?",
                                      "conversacion_id": accion["conversacion_id"]}).json()
respuesta = seguir(seguimiento["job_id"], hasta=("DONE", "FAILED"))
print("\n" + respuesta["resultado"]["informe"]["resumen"])
print("Fuentes:", respuesta["resultado"]["informe"]["fuentes"])

conversacion = api.get(f"/conversations/{accion['conversacion_id']}").json()
print(f"\nLa conversación tiene {len(conversacion['turnos'])} turnos:")
for turno in conversacion["turnos"]:
    print(f" - [{turno['referencia']}] {turno['pedido']}\n   -> {turno['resumen'][:140]}")

supervisor -> 

redactor -> validador -> 

supervisor -> 

[DONE] 10.151 s de proceso

Se gestionó la compensación para los clientes afectados por el incidente de pagos mediante el envío de cupones de descuento. La acción fue aprobada por Roger con un costo total estimado.
Fuentes: ['conversacion#turno-1']

La conversación tiene 2 turnos:
 - [conversacion#turno-1] Mandá un cupón del 10% a los clientes con cobro doble por el incidente de pagos.
   -> El incidente por caída de proveedor externo afectó a 1.240 pedidos y resultó en 312 cobros duplicados. Se procedió a enviar el cupón de comp
 - [conversacion#turno-2] ¿Qué acción se tomó con esos clientes y cuánto costó?
   -> Se gestionó la compensación para los clientes afectados por el incidente de pagos mediante el envío de cupones de descuento. La acción fue a


## Dónde mirar las trazas

En Phoenix (http://127.0.0.1:6006 → proyecto `colibri-intelligence`) cada pedido de este notebook es una traza: el supervisor, cada especialista, cada llamada al LLM y cada herramienta (incluida la búsqueda híbrida) con su latencia, sus tokens y su costo. Los dos turnos de la conversación quedan en la misma **sesión**, porque el `session.id` de la traza es el `conversacion_id`.